# HAMIC · TPU pretraining

Check the settings in the next cell, run the cells in order, pick a run configuration and press **Train**. After pushing new code, restart the kernel.

This is the JAX/TPU port of the training notebook; the cell order, controls and report match it.

In [ ]:
# Any Git ref works after "@": a branch (main), a tag or a commit SHA. Pin a SHA for
# a multi-session run so every session uses the same code (resume checks it).
HAMIC_SOURCE = "git+https://github.com/HaianCao/HAMIC-LM@main"
HUB_REPO = "HaianCao/hamic-runs"  # private model repo: checkpoints under runs/<run_id>/
DATA_REPO = "HaianCao/hamic-data"  # private dataset repo: tokenized shards
XET_HIGH_PERFORMANCE = True  # read at first huggingface_hub import; restart to change

In [ ]:
import json
import os as _os
import shutil
import sys
import tarfile
import tempfile
import urllib.request
from pathlib import Path

_os.environ["HF_XET_HIGH_PERFORMANCE"] = "1" if XET_HIGH_PERFORMANCE else "0"
requested_ref = HAMIC_SOURCE.rsplit("@", 1)[-1]
repo_url = HAMIC_SOURCE.removeprefix("git+").rsplit("@", 1)[0].removesuffix(".git")
owner_repo = repo_url.split("github.com/", 1)[-1]
api_url = f"https://api.github.com/repos/{owner_repo}"
work_root = Path(tempfile.gettempdir()) / "hamic-tpu"
source_dir = work_root / "source"
stamp = work_root / "source-ref.json"
with urllib.request.urlopen(f"{api_url}/commits/{requested_ref}", timeout=30) as response:
    installed_commit = json.load(response)["sha"]
stamp_data = json.loads(stamp.read_text()) if stamp.is_file() else None
if not isinstance(stamp_data, dict):
    stamp_data = None
if (
    not source_dir.is_dir()
    or stamp_data is None
    or stamp_data.get("ref") != requested_ref
    or stamp_data.get("commit") != installed_commit
):
    archive = work_root / "source.tar.gz"
    work_root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{repo_url}/archive/{installed_commit}.tar.gz", archive)
    unpack = work_root / "unpack"
    shutil.rmtree(unpack, ignore_errors=True)
    unpack.mkdir()
    with tarfile.open(archive) as tar:
        try:
            tar.extractall(unpack, filter="data")
        except TypeError:
            tar.extractall(unpack)
    (top,) = unpack.iterdir()
    shutil.rmtree(source_dir, ignore_errors=True)
    shutil.move(str(top), source_dir)
    shutil.rmtree(unpack, ignore_errors=True)
    archive.unlink()
    stamp.write_text(json.dumps({"ref": requested_ref, "commit": installed_commit}))
sys.path.insert(0, str(source_dir / "src"))
if "hamic_jax" in sys.modules:
    raise RuntimeError(
        "An older hamic_jax is already imported. Restart the kernel and run this notebook again."
    )
import hamic_jax

if Path(hamic_jax.__file__).resolve().parent != (source_dir / "src" / "hamic_jax").resolve():
    raise RuntimeError("hamic_jax was imported from outside the pinned source tree")
hamic_ready = installed_commit
print(f"`hamic_jax` installed from `@{requested_ref}` at commit `{installed_commit[:12]}`")

In [ ]:
import os

# Run configurations reference presets by repository-relative paths, so work from the
# checkout of exactly the installed commit.
os.chdir(source_dir)
configs_dir = source_dir / "configs"
run_configs = sorted(
    p.relative_to(source_dir).as_posix()
    for p in (configs_dir / "train").rglob("*.yaml")
)
print(f"Working directory `{source_dir}` · {len(run_configs)} run configurations")

In [ ]:
import jax

del hamic_ready  # dependency only: run after the install cell
backend = jax.default_backend()
if backend != "tpu":
    raise RuntimeError(
        f"No TPU attached (jax.default_backend()={backend!r}). Attach the TPU and rerun."
    )
devices = jax.devices()
print(
    f"TPU **{devices[0].device_kind}** · JAX `{jax.__version__}` · "
    f"{len(devices)} device(s) · code `{installed_commit[:12]}`"
)

In [ ]:
import getpass

hub_token = os.environ.get("HF_KEY", "").strip()
if not hub_token:
    hub_token = getpass.getpass("Hugging Face token (HF_KEY): ").strip()
if not hub_token:
    raise RuntimeError("Enter a Hugging Face token in HF_KEY or at the prompt.")
print("The token stays in this session's memory and is not saved in the notebook.")

In [ ]:
from hamic_jax.checkpoint import HuggingFaceObjectStore

run_store = HuggingFaceObjectStore(HUB_REPO, private=True, token=hub_token)
data_store = HuggingFaceObjectStore(
    DATA_REPO, repo_type="dataset", create=False, token=hub_token
)
account = run_store.whoami()
print(f"Hub account **{account}** · runs `{HUB_REPO}` · data `{DATA_REPO}`")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

default = next((c for c in run_configs if "proxy" in c), run_configs[0])
config_choice = widgets.Dropdown(
    options=run_configs, value=default, description="Run configuration"
)
mode = widgets.RadioButtons(
    options=[
        ("Test run (first steps only, nothing saved)", "bench"),
        ("Train / resume", "train"),
    ],
    value="bench",
    description="Mode",
)
# Test runs keep the full-length schedule and stop early, so their first steps match
# a real run with the same settings. The overrides below apply to test runs only.
test_steps = widgets.BoundedIntText(
    value=60, min=10, max=5000, step=10, description="Test steps"
)
test_lr = widgets.Text(value="", description="Test learning rate (empty: from config)")
test_warmup = widgets.Text(value="", description="Test warmup ratio (empty: from config)")
test_attention = widgets.Dropdown(
    options=["from config", "sdpa", "flex_attention"],
    value="from config",
    description="Test attention kernel",
)
display(
    widgets.VBox(
        [
            config_choice,
            mode,
            widgets.HBox([test_steps, test_lr, test_warmup, test_attention]),
        ]
    )
)
print("Set the controls above, then run the training cell.")

In [ ]:
from hamic_jax.lab import discover_training_configs
from hamic_jax.spec import read_yaml

candidates, rejected = discover_training_configs(source_dir / "configs" / "train")
catalog = {candidate.name: candidate for candidate in candidates}
if config_choice.value not in catalog:
    raise KeyError(f"{config_choice.value!r} is not in {sorted(catalog)}")
candidate = catalog[config_choice.value]
config, preset = candidate.config, candidate.preset
raw_config = read_yaml(source_dir / candidate.name)
summary = {
    "run_id": config.run_id,
    "model": f"{preset.name} ({preset.architecture})",
    "parameters": f"{preset.num_parameters / 1e6:.1f}M",
    "objective": config.objective
    + (f" (mask {config.mask_rate:.0%})" if config.objective == "mlm" else ""),
    "tokens": f"{config.total_tokens / 1e9:.3f}B",
    "steps": config.total_steps,
    "tokens_per_step": config.tokens_per_step,
    "tokenized": config.tokenized,
}
for key, value in summary.items():
    print(f"{key:>15}: {value}")
if rejected:
    print(f"Rejected: {rejected}")

In [ ]:
from hamic_jax.lab import fetch_corpus

config, corpus = fetch_corpus(config, data_store, work_root, config.run_id)
split_tokens = {name: value["tokens"] for name, value in corpus.manifest["splits"].items()}
epochs = config.total_tokens / split_tokens["train"]
print(f"Shards ready: `{split_tokens}` · the budget is **{epochs:.2f} epochs** of train")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

train_button = widgets.Button(description="Train", button_style="success")
train_requested = {"value": False}


def _request_train(_):
    train_requested["value"] = True


train_button.on_click(_request_train)
display(train_button)

In [ ]:
import dataclasses
import shutil

from tqdm.auto import tqdm

from hamic_jax.checkpoint import LocalObjectStore
from hamic_jax.lab import run_training_session
from hamic_jax.session import SessionLimits, run_session

if not train_requested["value"]:
    raise RuntimeError("Press **Train** to start.")

if mode.value == "bench":
    overrides = {}
    if test_lr.value.strip():
        overrides["learning_rate"] = float(test_lr.value)
    if test_warmup.value.strip():
        overrides["warmup_ratio"] = float(test_warmup.value)
    if test_attention.value != "from config":
        overrides["attention"] = test_attention.value
    run_config = dataclasses.replace(
        config,
        run_id=f"{config.run_id}-bench",
        eval_every_steps=0,
        checkpoint_every_steps=0,
        checkpoint_every_minutes=0.0,
        **overrides,
    )
    print(
        f"Test run: lr={run_config.learning_rate:g}, "
        f"warmup={run_config.warmup_steps} steps, attention={run_config.attention}, "
        f"stop after {int(test_steps.value)}",
        flush=True,
    )
    # Fresh store each time, so a test never resumes from an earlier test.
    shutil.rmtree(work_root / "bench-store", ignore_errors=True)
    store = LocalObjectStore(work_root / "bench-store")
    with tqdm(
        total=int(test_steps.value), desc=f"Bench {run_config.run_id}", unit="step"
    ) as bar:

        def progress(step, target, loss):
            bar.update(max(step - bar.n, 0))
            bar.set_postfix_str(f"loss {loss:.5f}")

        result = run_session(
            run_config,
            preset,
            corpus,
            store,
            work_root / "bench",
            SessionLimits(
                steps=int(test_steps.value),
                checkpoint_every_steps=0,
                checkpoint_every_seconds=0.0,
                log_every_steps=max(1, int(test_steps.value) // 5),
                eval_every_steps=0,
                checkpoint_at_end=False,
            ),
            source_commit=installed_commit,
            raw_config=dataclasses.asdict(run_config),
            progress=progress,
            log=print,
        )
else:
    with tqdm(
        total=config.total_steps, desc=f"Train {config.run_id}", unit="step"
    ) as bar:

        def progress(step, target, loss):
            bar.update(max(step - bar.n, 0))
            bar.set_postfix_str(f"loss {loss:.5f}")

        result = run_training_session(
            config,
            preset,
            corpus,
            run_store,
            work_root / "runs" / config.run_id,
            source_commit=installed_commit,
            raw_config=raw_config,
            progress=progress,
            log=print,
        )

# Later log windows only: the first ones include compilation and warmup.
steady = [
    record["tokens_per_second"]
    for record in result.metrics[len(result.metrics) // 2:]
    if "tokens_per_second" in record
]
tokens_per_second = sum(steady) / max(1, len(steady))
flops = 6 * preset.num_parameters * tokens_per_second
hours_left = (config.total_steps - result.final_step) * config.tokens_per_step
hours_left /= max(tokens_per_second, 1.0) * 3600
print(
    f"Status **{result.status}** · step `{result.final_step}/{config.total_steps}` · "
    f"resumed from `{result.resumed_from_step}` · published `{result.published_steps}`"
)
print(
    f"Throughput **{tokens_per_second:,.0f} tokens/s** (~{flops / 1e12:.0f} TFLOP/s "
    f"by 6·N·D) · about **{hours_left:.1f} h** of training left"
)
for warning in result.warnings:
    print(f"\n> Warning: {warning}")

In [ ]:
from IPython.display import display

rows = [
    {
        "step": step,
        "eval_loss": loss,
        "eval_bits_per_byte": result.eval_bits_per_byte.get(step),
    }
    for step, loss in sorted(result.evaluations.items())
]
if rows:
    import pandas as pd

    display(pd.DataFrame(rows))
else:
    print("No evaluations in this session.")